# Causal Inference Lab

Exploring causal inference methods: Simpson's Paradox, DAGs, propensity scores, and more.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import NearestNeighbors
from scipy import stats
np.random.seed(42)

## 1. Simpson's Paradox

In [ ]:
# Simpson's Paradox: A treatment appears harmful overall but is beneficial within each group
# Scenario: Drug trial with a confounding variable (severity)

n = 1000
# Severity is a confounder: affects both treatment assignment and outcome
severity = np.random.binomial(1, 0.5, n)  # 0=mild, 1=severe

# Severe patients are more likely to get treatment (confounding!)
treatment_prob = 0.3 + 0.5 * severity
treatment = np.random.binomial(1, treatment_prob)

# True effect: treatment HELPS (reduces bad outcome by 20%)
# But severity increases bad outcome by 40%
outcome_prob = 0.2 + 0.4 * severity - 0.2 * treatment
outcome_prob = np.clip(outcome_prob, 0.01, 0.99)
outcome = np.random.binomial(1, outcome_prob)  # 1 = bad outcome

df = pd.DataFrame({'severity': severity, 'treatment': treatment, 'outcome': outcome})

# Overall (misleading) comparison
overall_treated = df[df.treatment == 1].outcome.mean()
overall_control = df[df.treatment == 0].outcome.mean()

print("SIMPSON'S PARADOX DEMONSTRATION")
print("=" * 50)
print(f"\nOverall bad outcome rate:")
print(f"  Treated:   {overall_treated:.3f}")
print(f"  Control:   {overall_control:.3f}")
print(f"  Naive conclusion: Treatment INCREASES bad outcomes by {overall_treated - overall_control:.3f}!")

# Stratified (correct) comparison
print(f"\nStratified by severity:")
for sev in [0, 1]:
    sub = df[df.severity == sev]
    t_rate = sub[sub.treatment == 1].outcome.mean()
    c_rate = sub[sub.treatment == 0].outcome.mean()
    label = 'Mild' if sev == 0 else 'Severe'
    print(f"  {label}: Treated={t_rate:.3f}, Control={c_rate:.3f}, Effect={t_rate - c_rate:.3f}")

print(f"\nCorrect conclusion: Treatment DECREASES bad outcomes in BOTH groups!")
print(f"The paradox occurs because severity confounds the treatment-outcome relationship.")


In [ ]:
# Visualize Simpson's Paradox
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Overall view (misleading)
groups = ['Control', 'Treated']
overall_rates = [overall_control, overall_treated]
axes[0].bar(groups, overall_rates, color=['green', 'red'], alpha=0.7)
axes[0].set_ylabel('Bad Outcome Rate')
axes[0].set_title('Overall (Misleading): Treatment Looks Harmful')
axes[0].set_ylim(0, 0.7)
axes[0].grid(True, alpha=0.3, axis='y')

# Stratified view (correct)
x = np.arange(2)
width = 0.35
for sev, color, label in [(0, 'lightblue', 'Mild'), (1, 'salmon', 'Severe')]:
    sub = df[df.severity == sev]
    rates = [sub[sub.treatment == 0].outcome.mean(), sub[sub.treatment == 1].outcome.mean()]
    offset = -width/2 if sev == 0 else width/2
    axes[1].bar(x + offset, rates, width, label=label, color=color, edgecolor='black', linewidth=0.5)

axes[1].set_xticks(x)
axes[1].set_xticklabels(['Control', 'Treated'])
axes[1].set_ylabel('Bad Outcome Rate')
axes[1].set_title('Stratified (Correct): Treatment Helps Both Groups')
axes[1].legend()
axes[1].set_ylim(0, 0.7)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


## 2. Causal DAG (Directed Acyclic Graph)

In [ ]:
# Represent a causal DAG as an adjacency matrix
# Nodes: 0=Severity, 1=Treatment, 2=Outcome, 3=Age, 4=Recovery
node_names = ['Severity', 'Treatment', 'Outcome', 'Age', 'Recovery']
n_nodes = len(node_names)

# Adjacency matrix: adj[i][j] = 1 means i -> j (i causes j)
adj_matrix = np.zeros((n_nodes, n_nodes), dtype=int)
adj_matrix[0, 1] = 1  # Severity -> Treatment
adj_matrix[0, 2] = 1  # Severity -> Outcome
adj_matrix[1, 2] = 1  # Treatment -> Outcome (causal effect we want)
adj_matrix[3, 0] = 1  # Age -> Severity
adj_matrix[3, 1] = 1  # Age -> Treatment
adj_matrix[2, 4] = 1  # Outcome -> Recovery

print("Causal DAG Adjacency Matrix:")
dag_df = pd.DataFrame(adj_matrix, index=node_names, columns=node_names)
print(dag_df)

print("\nEdges:")
for i in range(n_nodes):
    for j in range(n_nodes):
        if adj_matrix[i, j]:
            print(f"  {node_names[i]} -> {node_names[j]}")

# Identify backdoor paths from Treatment to Outcome
print("\nBackdoor paths from Treatment to Outcome:")
print("  Treatment <- Severity -> Outcome")
print("  Treatment <- Age -> Severity -> Outcome")
print("\nTo identify causal effect: condition on {Severity} or {Age, Severity}")


## 3. Backdoor Adjustment

In [ ]:
# Backdoor adjustment formula:
# P(Y|do(T)) = sum_z P(Y|T,Z) * P(Z)
# where Z is the backdoor adjustment set

# Generate data from DAG
n = 5000
age = np.random.normal(50, 10, n)
severity = (age > 50).astype(int) + np.random.binomial(1, 0.3, n)
severity = np.clip(severity, 0, 1)

# Treatment assignment (confounded by severity)
treat_prob = 1 / (1 + np.exp(-(severity - 0.3 + 0.01 * (age - 50))))
treatment = np.random.binomial(1, treat_prob)

# True causal effect of treatment = -2.0
true_ate = -2.0
outcome = 5 + 3 * severity + 0.05 * age + true_ate * treatment + np.random.normal(0, 1, n)

print(f"True Average Treatment Effect (ATE): {true_ate}")

# Naive estimate (biased)
naive_ate = outcome[treatment == 1].mean() - outcome[treatment == 0].mean()
print(f"\nNaive ATE (biased): {naive_ate:.3f}")

# Backdoor adjustment: condition on severity
# Using linear regression with confounder
X = np.column_stack([treatment, severity, age])
reg = LinearRegression().fit(X, outcome)
adjusted_ate = reg.coef_[0]  # Coefficient on treatment
print(f"Backdoor-adjusted ATE (regression): {adjusted_ate:.3f}")
print(f"\nBias removed: {abs(naive_ate - true_ate) - abs(adjusted_ate - true_ate):.3f}")


## 4. Propensity Score Matching

In [ ]:
# Step 1: Estimate propensity scores (probability of treatment given confounders)
X_confounders = np.column_stack([severity, age])
ps_model = LogisticRegression()
ps_model.fit(X_confounders, treatment)
propensity_scores = ps_model.predict_proba(X_confounders)[:, 1]

print("Propensity Score Distribution")
print(f"  Treated group:  mean={propensity_scores[treatment==1].mean():.3f}, "
      f"std={propensity_scores[treatment==1].std():.3f}")
print(f"  Control group:  mean={propensity_scores[treatment==0].mean():.3f}, "
      f"std={propensity_scores[treatment==0].std():.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(propensity_scores[treatment==1], bins=30, alpha=0.5, label='Treated', color='red', density=True)
ax.hist(propensity_scores[treatment==0], bins=30, alpha=0.5, label='Control', color='blue', density=True)
ax.set_xlabel('Propensity Score')
ax.set_ylabel('Density')
ax.set_title('Propensity Score Distribution by Treatment Group')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Step 2: Match treated units to control units with similar propensity scores
treated_idx = np.where(treatment == 1)[0]
control_idx = np.where(treatment == 0)[0]

# Use nearest neighbor matching on propensity scores
nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
nn.fit(propensity_scores[control_idx].reshape(-1, 1))
distances, indices = nn.kneighbors(propensity_scores[treated_idx].reshape(-1, 1))

matched_control_idx = control_idx[indices.flatten()]

# Step 3: Estimate ATE from matched samples
ate_matched = outcome[treated_idx].mean() - outcome[matched_control_idx].mean()

print("Propensity Score Matching Results")
print("=" * 40)
print(f"  Matched pairs: {len(treated_idx)}")
print(f"  Mean matching distance: {distances.mean():.4f}")
print(f"\n  Naive ATE:    {naive_ate:.3f}")
print(f"  PSM ATE:      {ate_matched:.3f}")
print(f"  True ATE:     {true_ate:.3f}")
print(f"\n  PSM error:    {abs(ate_matched - true_ate):.3f}")
print(f"  Naive error:  {abs(naive_ate - true_ate):.3f}")


## 5. A/B Test Simulation

In [ ]:
# Demonstrate when simple A/B comparison fails due to selection bias
n = 2000

# User engagement score (confounder)
engagement = np.random.exponential(2, n)

# Non-random assignment: high-engagement users more likely to see new feature
# This simulates a biased "A/B test"
assign_prob = 1 / (1 + np.exp(-(engagement - 3)))
group_biased = np.random.binomial(1, assign_prob)  # Biased assignment
group_random = np.random.binomial(1, 0.5, n)       # True random assignment

# True effect of feature = +0.5 on conversion rate
true_effect = 0.5
base_conversion = 2 + 0.8 * engagement  # Engagement affects conversion

# Outcomes under biased assignment
outcome_biased = base_conversion + true_effect * group_biased + np.random.normal(0, 1, n)

# Outcomes under random assignment
outcome_random = base_conversion + true_effect * group_random + np.random.normal(0, 1, n)

# Compare estimates
biased_estimate = outcome_biased[group_biased==1].mean() - outcome_biased[group_biased==0].mean()
random_estimate = outcome_random[group_random==1].mean() - outcome_random[group_random==0].mean()

print("A/B Test: Random vs Biased Assignment")
print("=" * 50)
print(f"  True treatment effect: {true_effect:.3f}")
print(f"  Biased 'A/B test' estimate: {biased_estimate:.3f} (WRONG - confounded)")
print(f"  Proper randomized estimate: {random_estimate:.3f} (CORRECT)")
print(f"\n  Lesson: Without randomization, naive comparison is misleading!")


## 6. Instrumental Variables

In [ ]:
# IV example: Effect of education on earnings
# Instrument: Distance to college (affects education but not earnings directly)
n = 3000

# Unobserved confounder: ability
ability = np.random.normal(0, 1, n)

# Instrument: distance to college (random, uncorrelated with ability)
distance = np.random.uniform(0, 100, n)

# Education affected by distance (instrument) and ability (confounder)
education = 16 - 0.03 * distance + 1.5 * ability + np.random.normal(0, 1, n)

# Earnings affected by education and ability (confounder)
true_returns = 2.0  # True causal return to education
earnings = 20 + true_returns * education + 3 * ability + np.random.normal(0, 2, n)

# Naive OLS (biased - omits ability)
naive_reg = LinearRegression().fit(education.reshape(-1, 1), earnings)
naive_estimate = naive_reg.coef_[0]

# IV estimation (Two-Stage Least Squares)
# Stage 1: Regress education on instrument
stage1 = LinearRegression().fit(distance.reshape(-1, 1), education)
education_hat = stage1.predict(distance.reshape(-1, 1))

# Stage 2: Regress earnings on predicted education
stage2 = LinearRegression().fit(education_hat.reshape(-1, 1), earnings)
iv_estimate = stage2.coef_[0]

print("Instrumental Variables Estimation")
print("=" * 50)
print(f"  True causal effect of education on earnings: {true_returns:.3f}")
print(f"  Naive OLS estimate (biased): {naive_estimate:.3f}")
print(f"  IV estimate (2SLS): {iv_estimate:.3f}")
print(f"\n  First stage F-stat: {stats.pearsonr(distance, education)[0]**2 * n:.1f}")
print(f"  (Rule of thumb: F > 10 for strong instrument)")


## 7. Difference-in-Differences

In [ ]:
# DiD: Estimating effect of a policy change
# Scenario: Minimum wage increase in one state vs control state

n_periods = 20
treatment_time = 10  # Policy implemented at period 10

time = np.arange(n_periods)
true_effect = 3.0  # True policy effect

# Treatment group (state with policy change)
trend_treated = 20 + 0.5 * time + np.random.normal(0, 0.5, n_periods)
# Add treatment effect after intervention
trend_treated[treatment_time:] += true_effect

# Control group (parallel trend assumption)
trend_control = 18 + 0.5 * time + np.random.normal(0, 0.5, n_periods)

# DiD estimate
pre_diff = trend_treated[:treatment_time].mean() - trend_control[:treatment_time].mean()
post_diff = trend_treated[treatment_time:].mean() - trend_control[treatment_time:].mean()
did_estimate = post_diff - pre_diff

print(f"Difference-in-Differences Estimation")
print(f"={'='*50}")
print(f"  Pre-treatment difference: {pre_diff:.3f}")
print(f"  Post-treatment difference: {post_diff:.3f}")
print(f"  DiD estimate: {did_estimate:.3f}")
print(f"  True effect: {true_effect:.3f}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(time, trend_treated, 'r-o', markersize=4, label='Treatment group')
ax.plot(time, trend_control, 'b-s', markersize=4, label='Control group')
ax.axvline(x=treatment_time - 0.5, color='gray', linestyle='--', label='Intervention')

# Show counterfactual
counterfactual = trend_control[treatment_time:] + pre_diff
ax.plot(time[treatment_time:], counterfactual, 'r--', alpha=0.5, label='Counterfactual (no treatment)')

# Annotate the effect
mid_post = (treatment_time + n_periods) // 2
ax.annotate('', xy=(mid_post, trend_treated[mid_post]), 
            xytext=(mid_post, counterfactual[mid_post - treatment_time]),
            arrowprops=dict(arrowstyle='<->', color='green', lw=2))
ax.text(mid_post + 0.3, (trend_treated[mid_post] + counterfactual[mid_post - treatment_time])/2,
        f'Effect\n≈{did_estimate:.1f}', color='green', fontsize=10)

ax.set_xlabel('Time Period')
ax.set_ylabel('Outcome')
ax.set_title('Difference-in-Differences')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Comparison: Naive vs Causal Estimates

In [ ]:
# Summary comparison across all methods
methods = ['Naive\n(Correlation)', 'Backdoor\nAdjustment', 'Propensity\nScore Matching', 
           'Instrumental\nVariables', 'Diff-in-\nDifferences']
estimates = [naive_ate, adjusted_ate, ate_matched, iv_estimate, did_estimate]
true_effects = [true_ate, true_ate, true_ate, true_returns, true_effect]
errors = [abs(e - t) for e, t in zip(estimates, true_effects)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Estimates vs truth
x = np.arange(len(methods))
axes[0].bar(x, estimates, color='steelblue', alpha=0.7, label='Estimate')
axes[0].bar(x, true_effects, color='none', edgecolor='red', linewidth=2, label='True Effect')
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods, fontsize=8)
axes[0].set_ylabel('Effect Size')
axes[0].set_title('Causal Effect Estimates vs Truth')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Absolute errors
colors = ['red' if e > 0.5 else 'orange' if e > 0.2 else 'green' for e in errors]
axes[1].bar(x, errors, color=colors, alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods, fontsize=8)
axes[1].set_ylabel('Absolute Error')
axes[1].set_title('Estimation Error (lower is better)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKey Insight: Naive correlation often gives WRONG causal conclusions.")
print("Proper causal methods account for confounding and give estimates closer to truth.")


## 9. When to Use Each Method

| Method | Requirements | Strengths | Limitations |
|--------|-------------|-----------|-------------|
| Backdoor Adjustment | Know the DAG, measure confounders | Simple, efficient | Need correct DAG |
| Propensity Score Matching | Measure all confounders | Intuitive, non-parametric | Can't handle unmeasured confounding |
| Instrumental Variables | Valid instrument available | Handles unmeasured confounders | Instruments are rare, weak IV bias |
| Difference-in-Differences | Parallel trends assumption | Works with observational data | Requires good control group |
| RCT (A/B Test) | Can randomize | Gold standard | Expensive, sometimes unethical |

In [ ]:
# Final summary table
summary = pd.DataFrame({
    'Method': ['Naive Correlation', 'Backdoor Adjustment', 'PSM', 'IV (2SLS)', 'DiD'],
    'Estimate': [f'{e:.3f}' for e in estimates],
    'True Effect': [f'{t:.3f}' for t in true_effects],
    'Abs Error': [f'{e:.3f}' for e in errors],
    'Handles Confounding': ['No', 'Yes (observed)', 'Yes (observed)', 'Yes (unobserved)', 'Yes (time-invariant)']
})
print(summary.to_string(index=False))
